# Visualización atómica — Fórmula 1 (2016–2025)
Notebook reproducible en Google Colab. Lee la base limpia de Antonia Castillo, transforma porcentajes, crea el gráfico con Altair y exporta HTML interactivo + JPG.

## 1. Instalar librerías y subir CSV

In [ ]:
%pip -q install "altair>=5,<7" "vl-convert-python>=1.6" pillow pandas
from google.colab import files
from pathlib import Path
if not Path("AntoniaCastillo_DataBase_Limpia.csv").exists():
    files.upload()
assert Path("AntoniaCastillo_DataBase_Limpia.csv").exists(), "Sube el CSV indicado"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 8.8 MB/s eta 0:00:00


Saving AntoniaCastillo_DataBase_Limpia.csv to AntoniaCastillo_DataBase_Limpia.csv


## 2. Leer y preparar la base
Seleccionamos las diez temporadas 2016–2025. Convertimos porcentajes con `%` en números. Todos los resultados provienen del CSV.

In [ ]:
import pandas as pd
import altair as alt
import json, io
from PIL import Image
import vl_convert as vlc

df = pd.read_csv("AntoniaCastillo_DataBase_Limpia.csv")
campos = {"Temporada":"temporada", "Campeon Mundial - Piloto":"campeon", "Escuderia Piloto Campeon":"escuderia", "Victorias Campeon - Piloto":"victorias", "Total Carreras":"carreras", "Porcentaje Victorias Campeon - Piloto":"porcentaje"}
datos = df.loc[df["Temporada"].between(2016,2025),list(campos)].rename(columns=campos).copy()
datos["porcentaje"] = datos["porcentaje"].astype(str).str.replace("%","",regex=False).str.replace(",",".",regex=False).astype(float)
datos["resto"] = datos["carreras"] - datos["victorias"]
datos = datos.sort_values("temporada").reset_index(drop=True)
assert len(datos)==10
assert ((datos.victorias/datos.carreras*100 - datos.porcentaje).abs()<0.011).all()
display(datos)

,temporada,campeon,escuderia,victorias,carreras,porcentaje,resto
0,2016,Nico Rosberg,Mercedes,9,21,42.86,12
1,2017,Lewis Hamilton,Mercedes,9,20,45.00,11
2,2018,Lewis Hamilton,Mercedes,11,21,52.38,10
3,2019,Lewis Hamilton,Mercedes,11,21,52.38,10
4,2020,Lewis Hamilton,Mercedes,11,17,64.71,6
5,2021,Max Verstappen,Red Bull Racing Honda,10,22,45.45,12
6,2022,Max Verstappen,Red Bull Racing RBPT,15,22,68.18,7
7,2023,Max Verstappen,Red Bull Racing Honda RBPT,19,22,86.36,3
8,2024,Max Verstappen,Red Bull Racing Honda RBPT,9,24,37.50,15
9,2025,Lando Norris,McLaren,7,24,29.17,17


## 3. Crear la visualización original en Altair
Una línea roja, puntos por temporada y una referencia al 50%. Los puntos crecen al hacer clic y vuelven a su tamaño normal al deseleccionarlos.


In [ ]:
# Selección interactiva: amplía el punto de la temporada elegida.
seleccion = alt.selection_point(
    name="seleccion", fields=["temporada"], on="click", toggle=True,
    clear="click[event.item == null]"
)
base = alt.Chart(datos).encode(
    x=alt.X("temporada:O", title="Temporada", sort=list(range(2016,2026)), axis=alt.Axis(labelAngle=0,labelPadding=10)),
    y=alt.Y("porcentaje:Q", title="Victorias del campeón (%)", scale=alt.Scale(domain=[0,100]), axis=alt.Axis(values=[0,25,50,75,100],labelExpr="datum.value + '%'"))
)
linea = base.mark_line(color="#D64D4D",strokeWidth=3.5)
puntos = base.mark_circle(color="#D64D4D",stroke="white",strokeWidth=1.5).encode(
    size=alt.condition(seleccion, alt.value(330), alt.value(125)),
    tooltip=[
        alt.Tooltip("temporada:O",title="Temporada"), alt.Tooltip("campeon:N",title="Campeón"),
        alt.Tooltip("escuderia:N",title="Escudería"), alt.Tooltip("victorias:Q",title="Victorias"),
        alt.Tooltip("carreras:Q",title="Carreras disputadas"), alt.Tooltip("porcentaje:Q",title="Porcentaje",format=".2f")
    ]
).add_params(seleccion)
referencia=alt.Chart(pd.DataFrame({"porcentaje":[50]})).mark_rule(color="#9DA5AF",strokeDash=[6,6],strokeWidth=1.5).encode(y="porcentaje:Q")
grafico=(referencia+linea+puntos).properties(width=725,height=330,title=alt.TitleParams(text="No todos los campeones arrasan",subtitle="Porcentaje de carreras ganadas por el campeón mundial (2016–2025)",fontSize=23,subtitleFontSize=13,anchor="start",offset=18)).configure_view(stroke=None).configure_axis(grid=True,gridColor="#E7E9EE",domain=False,ticks=False,labelColor="#555D68",titleColor="#555D68").configure_title(color="#171C26",subtitleColor="#647080")
grafico


alt.LayerChart(...)

## 4. Exportar JPG
Guardamos imagen estática a partir de Altair para incluirla en la crónica.

In [ ]:
png_bytes=vlc.vegalite_to_png(grafico.to_json(),scale=2)
Image.open(io.BytesIO(png_bytes)).convert("RGB").save("vis_01.jpg",quality=95)
print("Imagen: vis_01.jpg")

Imagen: vis_01.jpg


## 5. Exportar HTML interactivo
El gráfico se genera con Altair. Al pulsar un punto, este crece y la ficha queda visible; una segunda pulsación o un clic fuera la cierra. Al iniciar no hay ningún año seleccionado.


In [ ]:
spec_json = json.dumps(grafico.to_dict(),ensure_ascii=False).replace("</","<\/")
html = """<!doctype html><html lang="es"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>No todos los campeones arrasan</title>
<script src="https://cdn.jsdelivr.net/npm/vega@6"></script><script src="https://cdn.jsdelivr.net/npm/vega-lite@6"></script><script src="https://cdn.jsdelivr.net/npm/vega-embed@7"></script>
<style>body{margin:0;padding:24px;background:#f5f6f8;color:#171c26;font-family:system-ui,Arial}main{background:white;border:1px solid #e4e6ec;border-radius:14px;padding:24px;max-width:960px;margin:auto}#vis{overflow-x:auto}.over{font-size:12px;letter-spacing:2px;color:#d64d4d;font-weight:800}#info{min-height:105px;border:1px solid #e4e6ec;border-radius:12px;padding:16px;box-sizing:border-box;display:flex;align-items:center;justify-content:center;color:#66717c}.record{display:block!important;color:#171c26!important}.top{display:flex;gap:20px;justify-content:space-between;flex-wrap:wrap}.name{font-weight:750;font-size:19px}.muted{color:#66717c;font-size:13px}.percentage{font-size:27px;color:#d64d4d;font-weight:800}.stats{border-top:1px solid #e4e6ec;padding-top:12px;margin-top:12px;font-size:14px}.legend{font-size:13px;color:#66717c;margin:12px 0 16px}.source{font-size:12px;color:#66717c;margin-top:15px}@media(max-width:600px){body{padding:0}main{border-radius:0;padding:14px}}</style></head><body><main>
<div class="over">FÓRMULA 1 · 2016–2025</div><div id="vis"></div>
<div class="legend"><span style="color:#d64d4d">━</span> Victorias del campeón (%)　　<span style="color:#9da5af">┄</span> Referencia del 50%</div>
<div id="info" aria-live="polite">↗ Pasa el cursor o toca un punto para conocer al campeón y sus victorias.</div>
<div class="source">Fuente: AntoniaCastillo_DataBase_Limpia.csv, temporadas 2016–2025.</div>
<script>const spec=__SPEC__; const info=document.getElementById('info');let pinned=false;let selectedYear=null;
function reset(){info.className='';info.textContent='↗ Pasa el cursor o toca un punto para conocer al campeón y sus victorias.'}
function render(d){if(!d || !d.campeon || !d.temporada)return;info.className='record';info.replaceChildren();const top=document.createElement('div');top.className='top';let a=document.createElement('div'),b=document.createElement('div');function put(parent,cls,s){let node=document.createElement('div');node.className=cls;node.textContent=s;parent.appendChild(node)}put(a,'muted',d.temporada+' · CAMPEÓN MUNDIAL');put(a,'name',d.campeon);put(a,'muted',d.escuderia);put(b,'percentage',Number(d.porcentaje).toLocaleString('es-CL',{minimumFractionDigits:2,maximumFractionDigits:2})+'%');put(b,'muted','de las carreras ganadas');top.append(a,b);info.append(top);put(info,'stats',d.victorias+' victorias de '+d.carreras+' carreras disputadas · '+d.resto+' ganadas por otros pilotos')}
vegaEmbed('#vis',spec,{actions:false,renderer:'svg'}).then(({view})=>{let valid=item=>item && item.datum && item.datum.campeon;
view.addEventListener('mouseover',(e,item)=>{if(!pinned&&valid(item))render(item.datum)});
view.addEventListener('mouseout',()=>{if(!pinned)reset()});
view.addEventListener('click',(e,item)=>{if(valid(item)){if(pinned&&selectedYear===item.datum.temporada){pinned=false;selectedYear=null;reset()}else{pinned=true;selectedYear=item.datum.temporada;render(item.datum)}}else{pinned=false;selectedYear=null;reset()}});
document.addEventListener('pointerdown',e=>{if(pinned&&!e.target.closest('#vis')){pinned=false;selectedYear=null;reset();view.change('seleccion_store',vega.changeset().remove(()=>true)).runAsync()}})
}).catch(err=>{console.error(err);info.textContent='Error de carga. Comprueba tu conexión a internet.'});
</script></main></body></html>"""
html=html.replace("__SPEC__",spec_json)
Path("vis_01.html").write_text(html,encoding="utf-8")
print("HTML: vis_01.html")

HTML: vis_01.html


<>:1: SyntaxWarning: invalid escape sequence '\/'
<>:1: SyntaxWarning: invalid escape sequence '\/'
/tmp/ipykernel_1395/1101749126.py:1: SyntaxWarning: invalid escape sequence '\/'
  spec_json = json.dumps(grafico.to_dict(),ensure_ascii=False).replace("</","<\/")


## 6. Descargar archivos para GitHub
El JPG y HTML van en `visualizacion/`; este notebook `.ipynb` va en `scripts/`; el CSV en la carpeta personal de la entrega.

In [ ]:
files.download("vis_01.html")
files.download("vis_01.jpg")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>